# Hexagon Segmentation Pipeline

Reads `02_merged_data.parquet` and assigns H3 hexagons at a configurable resolution

- `H3_RESOLUTION` — hexagon size (0 = coarsest, 15 = finest; 7≈5 km, 8≈1 km, 9≈0.3 km)

In [21]:
import pandas as pd
import polars as pl
import h3

## Load data

In [22]:
df = pl.read_parquet("../data/02_merged_data.parquet").to_pandas()
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
# df.head(3)

Shape: 478,114 rows x 34 columns


In [23]:
df.columns

Index(['trip_id', 'taxi_id', 'trip_start', 'trip_end', 'trip_seconds',
       'trip_miles', 'pickup_census_tract', 'dropoff_census_tract',
       'pickup_community_area', 'dropoff_community_area', 'fare_usd',
       'tips_usd', 'tolls_usd', 'extras_usd', 'trip_total_usd', 'payment_type',
       'company', 'pickup_lat', 'pickup_lon', 'dropoff_lat', 'dropoff_lon',
       'different_days', 'temperature_2m', 'rain', 'precipitation',
       'wind_speed_10m', 'wind_direction_10m', 'wind_gusts_10m',
       'apparent_temperature', 'relative_humidity_2m', 'snow_depth',
       'snowfall', 'temperature_2m_max', 'temperature_2m_min'],
      dtype='str')

# Assign H3 Hexagons

In [ ]:
def to_h3_cell(lat, lon):
    if pd.isna(lat) or pd.isna(lon):
        return None
    return h3.latlng_to_cell(float(lat), float(lon), 9)

df["start_h3_r9"] = [to_h3_cell(lat, lon) for lat, lon in zip(df["pickup_lat"],  df["pickup_lon"])]
df["end_h3_r9"]   = [to_h3_cell(lat, lon) for lat, lon in zip(df["dropoff_lat"], df["dropoff_lon"])]

print(f"  R9: {df['start_h3_r9'].nunique():>4} pickup cells, {df['end_h3_r9'].nunique():>4} dropoff cells")


In [28]:
df.columns

Index(['trip_id', 'taxi_id', 'trip_start', 'trip_end', 'trip_seconds',
       'trip_miles', 'pickup_census_tract', 'dropoff_census_tract',
       'pickup_community_area', 'dropoff_community_area', 'fare_usd',
       'tips_usd', 'tolls_usd', 'extras_usd', 'trip_total_usd', 'payment_type',
       'company', 'pickup_lat', 'pickup_lon', 'dropoff_lat', 'dropoff_lon',
       'different_days', 'temperature_2m', 'rain', 'precipitation',
       'wind_speed_10m', 'wind_direction_10m', 'wind_gusts_10m',
       'apparent_temperature', 'relative_humidity_2m', 'snow_depth',
       'snowfall', 'temperature_2m_max', 'temperature_2m_min', 'start_h3_r9',
       'end_h3_r9'],
      dtype='str')

# Save Parquet

In [29]:
df.to_parquet("../data/03_merged_data_with_h3.parquet", index=False)